<a href="https://colab.research.google.com/github/AntonioHNS/PPE-Detection-Benchmark/blob/main/PPE_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Configuração do Ambiente e Download do Dataset
!pip install ultralytics
!pip install roboflow
!pip install grad-cam
!pip uninstall -y tensorrt tensorrt-bindings tensorrt-cu12 tensorrt-cu12-bindings
!pip install "tensorrt<11.0"

import os
import glob
import cv2
import json
import pandas as pd
import matplotlib.pyplot as plt
import math
import torch
import random
import shutil
import time
import threading
import psutil
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import files
from google.colab import drive

drive.mount('/content/drive')
pasta_destino = '/content/drive/MyDrive/Pesos_TCC_v2'

rf = Roboflow(api_key="LPDv9Rps3hfsXMKg8k0T")
project = rf.workspace("ppe-656ud").project("ppe-fq2dk")
version = project.version(5)
dataset = version.download("yolov8")

# Outros projetos Roboflow comentados para referência:
#rf = Roboflow(api_key="LPDv9Rps3hfsXMKg8k0T")
#project = rf.workspace("hussnain-ahmed-xtgq8").project("chvg-dataset")
#version = project.version(1)
#dataset = version.download("yolov8")

#rf = Roboflow(api_key="LPDv9Rps3hfsXMKg8k0T")
#project = rf.workspace("ppedet-rghnf").project("ppe-tqmos")
#version = project.version(1)
#dataset = version.download("yolov8")

In [ ]:
#@title 2. Balanceamento e Divisão do Dataset

base_dir = "/content/PPE-5"

train_img_dir = os.path.join(base_dir, "train", "images")
train_lbl_dir = os.path.join(base_dir, "train", "labels")

valid_img_dir = os.path.join(base_dir, "valid", "images")
valid_lbl_dir = os.path.join(base_dir, "valid", "labels")

test_img_dir = os.path.join(base_dir, "test", "images")
test_lbl_dir = os.path.join(base_dir, "test", "labels")

qtd_mover_valid = 272
qtd_mover_test = 130

imagens_treino = [f for f in os.listdir(train_img_dir) if os.path.isfile(os.path.join(train_img_dir, f))]
random.seed(42) # Garante que a aleatoriedade seja reproduzível
random.shuffle(imagens_treino)

def mover_dados(lista_imagens, destino_img, destino_lbl):
    movidos = 0
    for img_nome in lista_imagens:
        # Mover imagem
        shutil.move(os.path.join(train_img_dir, img_nome), os.path.join(destino_img, img_nome))

        # Mover label correspondente (troca a extensão para .txt)
        nome_base = os.path.splitext(img_nome)[0]
        lbl_nome = nome_base + ".txt"
        lbl_origem = os.path.join(train_lbl_dir, lbl_nome)

        if os.path.exists(lbl_origem):
            shutil.move(lbl_origem, os.path.join(destino_lbl, lbl_nome))
        movidos += 1
    return movidos

print("Movendo imagens para validação...")
imagens_para_valid = imagens_treino[:qtd_mover_valid]
mover_dados(imagens_para_valid, valid_img_dir, valid_lbl_dir)

print("Movendo imagens para teste...")
imagens_para_test = imagens_treino[qtd_mover_valid : qtd_mover_valid + qtd_mover_test]
mover_dados(imagens_para_test, test_img_dir, test_lbl_dir)

print("\n" + "="*40)
print("NOVO BALANÇO DO DATASET:")
print(f"Treino:     {len(os.listdir(train_img_dir))} imagens")
print(f"Validação:  {len(os.listdir(valid_img_dir))} imagens")
print(f"Teste:      {len(os.listdir(test_img_dir))} imagens")
print("="*40)

In [4]:
#@title 3. Configurações de Treinamento e Funções de Monitoramento

dataset_yaml = f"{dataset.location}/data.yaml"


versoes = ['yolov8']
#versoes = ['yolov8', 'yolo11', 'yolo26']
tamanhos = ['n']
#tamanhos = ['n', 's', 'm', 'l']
precisoes = ['fp32']
#precisoes = ['int8', 'fp32', 'fp16']
resultados_benchmark = [] # Lista para armazenar os resultados de benchmark de cada modelo.

def medir_inferencia_com_memoria(modelo, dataset_path):


    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device=0)

    ram_base_bytes = psutil.virtual_memory().used

    stop_event = threading.Event()
    vram_samples = []
    ram_samples = []

    # Thread para monitorar o uso de memória em intervalos regulares durante a inferência.
    def monitor_memory():
        while not stop_event.is_set():
            if torch.cuda.is_available():
                vram_samples.append(torch.cuda.memory_allocated(device=0))
            ram_samples.append(psutil.virtual_memory().used)
            time.sleep(0.1) # Coleta a cada 0.1 segundo.

    t = threading.Thread(target=monitor_memory)
    t.start() # Inicia o monitoramento de memória.

    metrics = modelo.val(data=dataset_path, split='test', device=0)

    stop_event.set()
    t.join()


    pico_mb = 0
    if torch.cuda.is_available():
        pico_bytes = torch.cuda.max_memory_allocated(device=0)
        pico_mb = round(pico_bytes / (1024 * 1024), 2)

    avg_vram_mb = round(sum(vram_samples) / len(vram_samples) / (1024 * 1024), 2) if vram_samples else 0

    avg_ram_total_bytes = sum(ram_samples) / len(ram_samples) if ram_samples else ram_base_bytes
    avg_ram_mb = round(max(0, avg_ram_total_bytes - ram_base_bytes) / (1024 * 1024), 2)

    return metrics, pico_mb, avg_vram_mb, avg_ram_mb

In [ ]:
#@title 4. Treinamento dos Modelos Ultralytics

for versao in versoes:
    for tamanho in tamanhos:
        # Define o nome base do modelo e o nome do projeto para o salvamento dos resultados.
        nome_modelo_base = f"{versao}{tamanho}-seg.pt"
        nome_projeto = f"benchmark_seg_{versao}{tamanho}"
        print(f"\n{'='*60}\nIniciando Pipeline de Segmentação: {nome_modelo_base}\n{'='*60}")

        print(f"Treinando {nome_modelo_base}...")
        model = YOLO(nome_modelo_base)


        # Inicia o treinamento do modelo com as configurações definidas.
        # 'data': Caminho para o arquivo YAML do dataset.
        # 'epochs': Número de épocas de treinamento.
        # 'imgsz': Resolução da imagem de entrada.
        # 'batch': Tamanho do batch.
        # 'patience': Número de épocas sem melhoria para parar o treinamento (early stopping).
        # 'device': Dispositivo para treinamento (0 para GPU).
        # 'cls', 'cls_pw': Pesos para as perdas de classificação.
        # 'project', 'name': Diretórios de salvamento dos resultados no Google Drive.
        # 'cache': Habilita o cache de imagens para acelerar o treinamento.
        model.train(
            data= dataset_yaml,
            epochs=50,
            imgsz=640,
            batch=8,
            patience=10,
            device=0,
            cls=2.0,
            cls_pw=1.0,
            project=pasta_destino,
            name=nome_projeto,
            cache=True
        )

        # Define o caminho onde os melhores pesos do modelo treinado são salvos no Google Drive.
        #caminho_best_pt = f"runs/segment/resultados_tcc/{nome_projeto}/weights/best.pt"

        caminho_best_pt = f"{pasta_destino}/{nome_projeto}/weights/best.pt"
        print(f"[SUCESSO] Treinamento concluído. Tudo salvo em: {caminho_best_pt}")

In [ ]:
#@title 5. Avaliação e Benchmark de Modelos por Precisão

resultados_benchmark = []

for versao in versoes:
    for tamanho in tamanhos:
        # Define o nome do projeto e o caminho para os melhores pesos do modelo no Google Drive.
        nome_projeto = f"benchmark_seg_{versao}{tamanho}"
        caminho_best_pt = f"{pasta_destino}/{nome_projeto}/weights/best.pt"

        # Verifica se os pesos do modelo existem. Caso contrário, pula para o próximo modelo.
        if not os.path.exists(caminho_best_pt):
            print(f"[PULANDO] Pesos não encontrados para {versao.upper()} ({tamanho.upper()})")
            continue

        print(f"\n{'='*60}\nCorrigindo Validações de: {versao.upper()} ({tamanho.upper()})\n{'='*60}")

        for precisao in precisoes:
            print(f"\n---> Avaliando precisão: {precisao.upper()} <---")

            # Carrega o modelo YOLO com os pesos treinados.
            modelo_base = YOLO(caminho_best_pt)

            try:
                if precisao == 'fp32':
                    # Para FP32, executa a inferência diretamente com a função de medição de memória.
                    metrics, vram_mb, avg_vram_mb, avg_ram_mb = medir_inferencia_com_memoria(modelo_base, dataset_yaml)

                elif precisao == 'fp16':
                    # Para FP16, exporta o modelo para ONNX com precisão de 16 bits (half=True) e otimizado para GPU.
                    print("Exportando para ONNX FP16 acelerado por GPU...")
                    caminho_export = modelo_base.export(format='onnx', half=True, device=0)
                    modelo_fp16 = YOLO(caminho_export, task='segment')
                    metrics, vram_mb, avg_vram_mb, avg_ram_mb = medir_inferencia_com_memoria(modelo_fp16, dataset_yaml)

                elif precisao == 'int8':
                    # Para INT8, exporta o modelo para TensorRT INT8, com calibração usando o dataset.
                    caminho_export = modelo_base.export(format="engine", int8=True, imgsz=640,  data=dataset_yaml, device=0)
                    modelo_int8 = YOLO(caminho_export)
                    metrics, vram_mb, avg_vram_mb, avg_ram_mb = medir_inferencia_com_memoria(modelo_int8, dataset_yaml)

                resultados_benchmark.append({
                    "Modelo": f"{versao.upper()}-SEG",
                    "Tamanho": tamanho.upper(),
                    "Precisao": precisao.upper(),
                    "mAP50_Box": round(metrics.box.map50, 4),
                    "mAP50_Mask": round(metrics.seg.map50, 4),
                    "mAP50-95_Mask": round(metrics.seg.map, 4),
                    "Latencia_ms": round(metrics.speed['inference'], 2),
                    "VRAM_Pico_MB": vram_mb,
                    "VRAM_Media_MB": avg_vram_mb,
                    "RAM_Media_MB": avg_ram_mb
                })

                print(f"[SUCESSO] {precisao.upper()}: mAP50_Mask={round(metrics.seg.map50, 4)} | Latência Corrigida={round(metrics.speed['inference'], 2)}ms")

            except Exception as e:
                # Captura e imprime erros que possam ocorrer durante o processo de avaliação.
                print(f"[ERRO] Falha crítica ao processar {versao}{tamanho} em {precisao}: {e}")

print("\n" + "="*60 + "\nCONSOLIDADO FINAL CORRIGIDO\n" + "="*60)
# Imprime os resultados de benchmark em formato JSON.
print(json.dumps(resultados_benchmark, indent=4))

# Converte os resultados para um DataFrame do pandas e salva em um arquivo CSV.
df = pd.DataFrame(resultados_benchmark)
df.to_csv("resultados_benchmark_segmentacao_oficial.csv", index=False)
print("\n[OK] Benchmark definitivo salvo em 'resultados_benchmark_segmentacao_oficial.csv'!")